In [9]:
import pandas as pd

df = pd.read_excel(
    "C:/Users/husnu/Downloads/Имя_клиента_stock_9_2026-04.xlsx",
    header=1
)

# kolonları temizle
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print(df.head())
print(df.info())

  номенклатура    поставщик              производитель  количество_остатков  \
0    Продукт 1  Поставщик 1  ООО Производитель номер 1                    1   
1    Продукт 2  Поставщик 2  ООО Производитель номер 1                    1   
2    Продукт 3  Поставщик 2  ООО Производитель номер 1                    1   
3    Продукт 1  Поставщик 1  ООО Производитель номер 1                    1   
4    Продукт 2  Поставщик 2  ООО Производитель номер 1                    1   

                   адрес  
0  улица "адрес" дом № 1  
1  улица "адрес" дом № 1  
2  улица "адрес" дом № 1  
3  улица "адрес" дом № 2  
4  улица "адрес" дом № 2  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   номенклатура         9 non-null      object
 1   поставщик            9 non-null      object
 2   производитель        9 non-null      object
 3   количество_ос

In [10]:
df.head(10)

,номенклатура,поставщик,производитель,количество_остатков,адрес
0,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,"улица ""адрес"" дом № 1"
1,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 1"
2,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 1"
3,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,"улица ""адрес"" дом № 2"
4,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 2"
5,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 2"
6,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,"улица ""адрес"" дом № 3"
7,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 3"
8,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,"улица ""адрес"" дом № 3"


In [11]:
import pandas as pd
import re

# 1) string kolonları genel temizle
str_cols = df.select_dtypes(include="object").columns

for col in str_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# 2) adres kolonunu temizle
df["address_clean"] = (
    df["адрес"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace('"', "", regex=False)   # tırnakları kaldır
)

# 3) ev numarasını ayır
df["house_number"] = df["address_clean"].str.extract(r"дом\s*№?\s*(\d+)", expand=False)

# 4) sokak kısmını ayır
df["street"] = (
    df["address_clean"]
    .str.replace(r"\s*дом\s*№?\s*\d+\s*$", "", regex=True)
    .str.strip()
)

# 5) house_number numeric yap
df["house_number"] = pd.to_numeric(df["house_number"], errors="coerce")

# 6) quantity numeric yap
df["количество_остатков"] = pd.to_numeric(df["количество_остатков"], errors="coerce")

# 7) type check
print(df.dtypes)
print("\n")
print(df.head())

номенклатура           object
поставщик              object
производитель          object
количество_остатков     int64
адрес                  object
address_clean          object
house_number            int64
street                 object
dtype: object


  номенклатура    поставщик              производитель  количество_остатков  \
0    Продукт 1  Поставщик 1  ООО Производитель номер 1                    1   
1    Продукт 2  Поставщик 2  ООО Производитель номер 1                    1   
2    Продукт 3  Поставщик 2  ООО Производитель номер 1                    1   
3    Продукт 1  Поставщик 1  ООО Производитель номер 1                    1   
4    Продукт 2  Поставщик 2  ООО Производитель номер 1                    1   

                   адрес        address_clean  house_number       street  
0  улица "адрес" дом № 1  улица адрес дом № 1             1  улица адрес  
1  улица "адрес" дом № 1  улица адрес дом № 1             1  улица адрес  
2  улица "адрес" дом № 1  улица адрес дом № 

In [12]:
cat_cols = ["номенклатура", "поставщик", "производитель", "street"]

for col in cat_cols:
    df[col] = df[col].astype("category")

print(df.dtypes)

номенклатура           category
поставщик              category
производитель          category
количество_остатков       int64
адрес                    object
address_clean            object
house_number              int64
street                 category
dtype: object


In [14]:
df = df.rename(columns={
    "address_clean": "очищенный_адрес",
    "house_number": "номер_дома",
    "street": "улица"
})

In [17]:
df = df.drop(columns=["очищенный_адрес"])

In [18]:
df["улица"] = df["улица"].str.replace(r"^улица\s+", "", regex=True)

In [22]:
df = df.drop(columns=["адрес"])

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   номенклатура         9 non-null      category
 1   поставщик            9 non-null      category
 2   производитель        9 non-null      category
 3   количество_остатков  9 non-null      int64   
 4   номер_дома           9 non-null      int64   
 5   улица                9 non-null      object  
dtypes: category(3), int64(2), object(1)
memory usage: 747.0+ bytes


In [25]:
df["улица"] = df["улица"].astype("category")

In [26]:
print(df.duplicated().sum())

0


In [27]:
print(df["номер_дома"].unique())
print(df["улица"].unique())

[1 2 3]
['адрес']
Categories (1, object): ['адрес']


In [28]:
df.head(10)

,номенклатура,поставщик,производитель,количество_остатков,номер_дома,улица
0,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,1,адрес
1,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,1,адрес
2,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,1,адрес
3,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,2,адрес
4,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,2,адрес
5,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,2,адрес
6,Продукт 1,Поставщик 1,ООО Производитель номер 1,1,3,адрес
7,Продукт 2,Поставщик 2,ООО Производитель номер 1,1,3,адрес
8,Продукт 3,Поставщик 2,ООО Производитель номер 1,1,3,адрес
